# TESTING

In [1]:
def needs_effectiveness_backfill(payload):
    """
    Returns True if at least one effectiveness metric is missing or NaN.
    """

    effectiveness = payload.get("effectiveness_metrics", {})

    for metric_name in effectiveness_metrics:
        if metric_name not in effectiveness:
            return True

        if pd.isna(effectiveness[metric_name]):
            return True

    return False

In [ ]:
import gc
import pickle
import re
import numpy as np
import pandas as pd
from pathlib import Path
from src import util

# ============================================================
# Metric definitions
# ============================================================

effectiveness_metrics = [
    "accuracy",
    "hamming_score",
    "f1_example",
    "f1_micro",
    "f1_macro",
]

drift_metrics = [
    "n_drifts_detected",
    "delay_for_detection",
    "missed_detection_rate",
]

time_metrics = [
    "time",
    "time_per_sample",
]

ALL_METRICS = effectiveness_metrics + drift_metrics + time_metrics

# Adjust these function names if your util.py uses slightly different names.
EFFECTIVENESS_FUNCTIONS = {
    "accuracy": util.accuracy_example,
    "hamming_score": util.hamming_score,
    "f1_example": util.f1_example,
    "f1_micro": util.f1_micro,
    "f1_macro": util.f1_macro,
}


# ============================================================
# Result-family classification
# ============================================================

ABLATION_EXACT_METHOD_PATTERN = re.compile(
    r"^(?:C3M_MC_(?:No)?NVI|LACE-(?:Agglo|C3M|KMeans)_best)$"
)
ABLATION_NOMC_SIM_PATTERN = re.compile(
    r"^C3M_NoMC_(?:No)?NVI_sim\d{3}$"
)


def classify_result_family(method_name: str):
    """
    Returns one of:
        - "ablation"
        - "hyperparameter_sensitivity"
        - "baseline"
        - None, meaning ignore this file
    """

    # Hyperparameter sensitivity:
    # e.g. LACE-C3M_w500_000_050_025NVI
    if method_name.startswith("LACE-C3M_w"):
        return "hyperparameter_sensitivity"

    # Baselines:
    # e.g. LD3, ADWIN, DDM, KSWIN, etc.
    # User definition: none contains C3M or LACE.
    if "C3M" not in method_name and "LACE" not in method_name:
        return "baseline"

    # Ablation. NoMC originals are intentionally excluded; only
    # C3M_NoMC_(No)NVI_simNNN variants belong here.
    if (
        ABLATION_EXACT_METHOD_PATTERN.fullmatch(method_name)
        or ABLATION_NOMC_SIM_PATTERN.fullmatch(method_name)
    ):
        return "ablation"

    return None


# ============================================================
# Helpers
# ============================================================

def get_existing_metric(payload, metric_name):
    """
    Read a metric from the pickle payload if it already exists.
    Returns np.nan if unavailable.
    """

    if metric_name in effectiveness_metrics:
        return payload.get("effectiveness_metrics", {}).get(metric_name, np.nan)

    if metric_name in drift_metrics:
        return payload.get("drift_metrics", {}).get(metric_name, np.nan)

    if metric_name == "time":
        return payload.get("time", {}).get("seconds", np.nan)

    if metric_name == "time_per_sample":
        time_info = payload.get("time", {})
        seconds = time_info.get("seconds", np.nan)
        samples_seen = time_info.get("stream_samples_seen", np.nan)

        if (
            pd.notna(seconds)
            and pd.notna(samples_seen)
            and samples_seen not in [0, None]
        ):
            return seconds / samples_seen

        return np.nan

    return np.nan


def backfill_missing_effectiveness_metrics(
    prediction_file,
    payload,
    y_true,
    dataset,
    method,
):
    """
    If effectiveness_metrics are missing from the pickle, compute them
    from y_true and payload["predictions"], then write them back into
    the same pickle using payload["effectiveness_metrics"].
    """

    if "effectiveness_metrics" not in payload:
        payload["effectiveness_metrics"] = {}

    missing_metrics = [
        metric_name
        for metric_name in effectiveness_metrics
        if metric_name not in payload["effectiveness_metrics"]
        or pd.isna(payload["effectiveness_metrics"][metric_name])
    ]

    if not missing_metrics:
        return payload, False

    if "predictions" not in payload:
        raise KeyError(
            f"{dataset}/{method}: missing effectiveness metrics, "
            f"but payload has no 'predictions' field."
        )

    y_pred = np.asarray(payload["predictions"], dtype=np.int8)

    if y_pred.ndim != 2:
        raise ValueError(
            f"{dataset}/{method}: invalid prediction shape {y_pred.shape}"
        )

    y_true_eval = y_true[:len(y_pred)]

    if y_true_eval.shape != y_pred.shape:
        raise ValueError(
            f"{dataset}/{method}: truth {y_true_eval.shape}, "
            f"predictions {y_pred.shape}"
        )

    for metric_name in missing_metrics:
        metric_function = EFFECTIVENESS_FUNCTIONS[metric_name]
        payload["effectiveness_metrics"][metric_name] = metric_function(
            y_true_eval,
            y_pred,
        )

    with prediction_file.open("wb") as file:
        pickle.dump(payload, file)

    return payload, True


# ============================================================
# Parse, backfill, collect rows
# ============================================================

rows_by_family = {
    "ablation": [],
    "hyperparameter_sensitivity": [],
    "baseline": [],
}

# Define these BEFORE the loop
PREDICTION_ROOT = Path("outputs/predictions")
CSV_ROOT = Path("outputs/csv")
CSV_ROOT.mkdir(parents=True, exist_ok=True)


for dataset_dir in sorted(PREDICTION_ROOT.iterdir()):
    if not dataset_dir.is_dir():
        continue

    dataset = dataset_dir.name
    prediction_files = sorted(dataset_dir.glob("*.pkl"))

    if dataset not in dataset_specs or not prediction_files:
        continue

    selected_files = []

    for prediction_file in prediction_files:
        method = prediction_file.stem
        family = classify_result_family(method)

        if family is not None:
            selected_files.append((prediction_file, method, family))

    if not selected_files:
        continue

    print(f"Processing {dataset}: {len(selected_files)} selected files")

    # ------------------------------------------------------------
    # First pass:
    # Check whether this dataset actually needs y_true loading.
    # ------------------------------------------------------------

    dataset_needs_y_true = False

    for prediction_file, method, family in selected_files:
        with prediction_file.open("rb") as file:
            payload = pickle.load(file)

        if needs_effectiveness_backfill(payload):
            dataset_needs_y_true = True
            break

        del payload

    # ------------------------------------------------------------
    # Load y_true only if needed.
    # ------------------------------------------------------------

    y_true = None

    if dataset_needs_y_true:
        print(f"  loading true labels for missing metrics: {dataset}")

        temporary_predictions, y_true, _, _, _ = util.load_predictions_and_labels(
            dataset_name=dataset,
            dataset_specs=dataset_specs,
            predictions_dir=PREDICTION_ROOT,
            filter_methods=[selected_files[0][1]],
            label_location="start",
        )

        del temporary_predictions
        y_true = np.asarray(y_true, dtype=np.int8)
        gc.collect()
    else:
        print(f"  all effectiveness metrics already exist: {dataset}")

    # ------------------------------------------------------------
    # Second pass:
    # Parse metrics and backfill only if needed.
    # ------------------------------------------------------------

    for index, (prediction_file, method, family) in enumerate(selected_files, start=1):
        with prediction_file.open("rb") as file:
            payload = pickle.load(file)

        method = prediction_file.stem

        if needs_effectiveness_backfill(payload):
            if y_true is None:
                raise RuntimeError(
                    f"{dataset}/{method}: missing effectiveness metrics, "
                    f"but y_true was not loaded."
                )

            payload, updated = backfill_missing_effectiveness_metrics(
                prediction_file=prediction_file,
                payload=payload,
                y_true=y_true,
                dataset=dataset,
                method=method,
            )

            if updated:
                print(f"  updated missing effectiveness metrics: {dataset}/{method}")

        row = {
            "dataset": dataset,
            "method": method,
        }

        for metric_name in ALL_METRICS:
            row[metric_name] = get_existing_metric(payload, metric_name)

        rows_by_family[family].append(row)

        del payload

        if index % 25 == 0:
            print(f"  {index}/{len(selected_files)}")
            gc.collect()

    if y_true is not None:
        del y_true

    gc.collect()


# ============================================================
# Save pivoted CSVs:
# /ablation/accuracy.csv
# /hyperparameter_sensitivity/accuracy.csv
# /baseline/accuracy.csv
#
# Each CSV:
# rows    = methods
# columns = datasets
# cells   = metric values
# ============================================================


PREDICTION_ROOT = Path("outputs/predictions")
CSV_ROOT = Path("outputs/csv")
CSV_ROOT.mkdir(parents=True, exist_ok=True)

for family, rows in rows_by_family.items():
    family_dir = CSV_ROOT / family
    family_dir.mkdir(parents=True, exist_ok=True)

    if not rows:
        print(f"No rows for {family}")
        continue

    metrics_long = pd.DataFrame(rows).sort_values(
        ["method", "dataset"]
    ).reset_index(drop=True)

    # Optional long-format backup.
    metrics_long.to_csv(
        family_dir / "all_metrics_long.csv",
        index=False,
    )

    for metric_name in ALL_METRICS:

        metric_table = metrics_long.pivot_table(
            index="method",
            columns="dataset",
            values=metric_name,
            aggfunc="first",
        )

        # Optional: keep dataset order from dataset_specs instead of alphabetical.
        ordered_columns = [
            dataset_name
            for dataset_name in dataset_specs.keys()
            if dataset_name in metric_table.columns
        ]
        metric_table = metric_table.reindex(columns=ordered_columns)

        metric_table.to_csv(family_dir / f"{metric_name}.csv")

print("Done.")